# 04 Cross-Dataset Orchestrator

Runs experiments across all datasets and aggregates results into cross-dataset tables and plots.

## 1) Setup

In [ ]:
import os
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
FIG_DIR = ROOT / 'outputs' / 'figures'
RES_DIR = ROOT / 'outputs' / 'results'
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

ALL_DATASETS = ['ml-100k', 'ml-1m', 'lastfm-2k', 'book-crossing']


## 2) Choose datasets

In [ ]:
DATASETS = ALL_DATASETS  # or subset like ['ml-100k', 'ml-1m']
TEST_SIZE = 0.2


## 3) Run all experiments for each dataset (calls `run_all.py`)

In [ ]:
for ds in DATASETS:
    cmd = ['python', str(ROOT / 'run_all.py'), '--dataset', ds, '--test-size', str(TEST_SIZE)]
    print('Running', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)


## 4) Aggregate per-dataset CSVs

In [ ]:
def read_if_exists(p: Path):
    return pd.read_csv(p) if p.exists() else None

macro_rows = []
fair_rows = []
fb_rows = []

for ds in DATASETS:
    m = read_if_exists(RES_DIR / f'macro_bias_metrics_{ds}.csv')
    if m is not None:
        m['dataset'] = ds
        macro_rows.append(m)

    f = read_if_exists(RES_DIR / f'user_centric_metrics_{ds}.csv')
    if f is not None:
        f['dataset'] = ds
        fair_rows.append(f)

    fb = read_if_exists(RES_DIR / f'feedback_simulation_metrics_{ds}.csv')
    if fb is not None:
        fb['dataset'] = ds
        fb_rows.append(fb)

macro = pd.concat(macro_rows, ignore_index=True) if macro_rows else None
fairness = pd.concat(fair_rows, ignore_index=True) if fair_rows else None
feedback = pd.concat(fb_rows, ignore_index=True) if fb_rows else None

macro, fairness, feedback


## 5) Save combined CSVs

In [ ]:
if macro is not None:
    macro.to_csv(RES_DIR / 'combined_macro_bias_metrics.csv', index=False)
if fairness is not None:
    fairness.to_csv(RES_DIR / 'combined_user_centric_metrics.csv', index=False)
if feedback is not None:
    feedback.to_csv(RES_DIR / 'combined_feedback_simulation_metrics.csv', index=False)

print('Saved combined CSVs to outputs/results')


## 6) Cross-dataset plots

In [ ]:
if macro is not None:
    macro_metrics = [
        'gini', 'arp', 'catalog_coverage', 'spearman_pop_vs_recfreq',
        'avg_popularity_percentile', 'long_tail_share_20pct',
        'aggregate_diversity_unique_items', 'metadata_token_entropy_bits',
    ]
    macro_metrics = [m for m in macro_metrics if m in macro.columns]
    for metric in macro_metrics:
        pivot = macro.pivot_table(index='model', columns='dataset', values=metric, aggfunc='mean')
        plt.figure(figsize=(10, 5))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
        plt.title(f'Macro-bias: {metric} (model × dataset)')
        plt.tight_layout()
        out = FIG_DIR / f'combined_macro_{metric}_heatmap.png'
        plt.savefig(out, dpi=180, bbox_inches='tight')
        plt.show()

if fairness is not None:
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=fairness,
        x='dataset',
        y='delta_accuracy_rmse_niche_minus_mainstream',
        hue='model',
    )
    plt.title('User-centric fairness gap (?RMSE = RMSE_Niche - RMSE_Mainstream)')
    plt.xticks(rotation=20)
    plt.tight_layout()
    out = FIG_DIR / 'combined_fairness_gap_delta_rmse.png'
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()

if feedback is not None:
    plt.figure(figsize=(12, 6))
    sns.lineplot(
        data=feedback,
        x='iteration',
        y='aggregate_diversity',
        hue='model',
        style='dataset',
        markers=True,
        dashes=False,
    )
    plt.title('Feedback simulation: diversity decay (all datasets)')
    plt.tight_layout()
    out = FIG_DIR / 'combined_feedback_diversity_decay.png'
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
